# D2 DLinear: запуск «как есть» в Colab

Ноутбук прогоняет авторский код без изменений на T4:

1. **FEDformer** (код из `upstream/FEDformer`), ETTh1, multivariate, L = 96, T = 96. В статье: MSE 0.376, MAE 0.419.
2. **DLinear** (`upstream/run_longExp.py`), ETTh1, L = 336, T = 96, конфиг из `configs/etth1-336-96.json`. В статье: MSE 0.375, MAE 0.399; на маке у нас 0.3841 / 0.4047.

Отклонения от авторских скриптов (все записываются в DECISIONS.md):

- `--features M` вместо `S` для FEDformer на ETTh1: в `FEDformer/scripts/LongForecasting.sh` стоит `S`, а Table 2 статьи multivariate.
- Свежие torch и NumPy из Colab вместо torch 1.9.0: под Python Colab сборок 1.9.0 нет.
- NumPy 2 удалил `np.Inf`, который использует авторский `EarlyStopping`. Код не правим: запускаем через `compat_run.py`, который до старта скрипта добавляет `np.Inf = np.inf`.

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import platform, sys
import numpy, pandas, torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available())
print('numpy', numpy.__version__, 'pandas', pandas.__version__)
print(platform.platform())

## Настройки

`SAVE_TO_DRIVE = True` сохранит папки запусков на Google Drive (колаб попросит доступ). Ключи ClearML берутся из «Секретов» колаба (иконка ключа слева): `CLEARML_API_ACCESS_KEY` и `CLEARML_API_SECRET_KEY`. Если их нет, клетка ClearML пропускается.

In [ ]:
SAVE_TO_DRIVE = False
USE_CLEARML = True
DRIVE_DIR = '/content/drive/MyDrive/dlinear-reproduction/runs'
REPO_URL = 'https://github.com/AntonGra/dlinear-reproduction.git'
UPSTREAM_COMMIT = '0c113668a3b88c4c4ee586b8c5ec3e539c4de5a6'

## Репозиторий и данные

In [ ]:
%cd /content
!rm -rf dlinear-reproduction
!git clone -q --recurse-submodules {REPO_URL}
%cd /content/dlinear-reproduction
!git log -1 --format='repo commit: %H %ad' --date=iso
!git -C upstream rev-parse HEAD
!git -C upstream diff --exit-code --stat && echo 'upstream: без изменений'

In [ ]:
import hashlib, json, os, subprocess, urllib.request

ROOT = '/content/dlinear-reproduction'
assert subprocess.check_output(['git', '-C', ROOT + '/upstream', 'rev-parse', 'HEAD'], text=True).strip() == UPSTREAM_COMMIT

source = json.load(open(ROOT + '/records/data-source.json'))
os.makedirs(ROOT + '/data', exist_ok=True)
for item in source['files']:
    path = os.path.join(ROOT, item['path'])
    if not os.path.exists(path):
        urllib.request.urlretrieve(item['url'], path)
    digest = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert digest == item['sha256'], item['path']
    print('ok', item['path'])

In [ ]:
!pip -q install einops clearml

In [ ]:
compat = """import os
import runpy
import sys

import numpy as np

if not hasattr(np, 'Inf'):
    np.Inf = np.inf

script = os.path.abspath(sys.argv[1])
sys.argv = sys.argv[1:]
sys.path.insert(0, os.path.dirname(script))
runpy.run_path(script, run_name='__main__')
"""
open('/content/compat_run.py', 'w').write(compat)

## ClearML

Один запуск = один Task в проекте `D2 DLinear`. Авторский код ничего не знает про ClearML: после прогона клетка разбирает его лог и отправляет лоссы по эпохам, итоговые метрики и лог.

In [ ]:
clearml_ok = False
if USE_CLEARML:
    try:
        from google.colab import userdata
        os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_API_ACCESS_KEY')
        os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_API_SECRET_KEY')
        os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
        os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml'
        os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
        clearml_ok = True
        print('clearml: ключи найдены')
    except Exception as e:
        print('clearml: пропускаем,', type(e).__name__)

## Запуск

In [ ]:
import re, time

EPOCH_RE = re.compile(r'Epoch: (\d+), Steps: \d+ \| Train Loss: ([\d.]+) Vali Loss: ([\d.]+) Test Loss: ([\d.]+)')
METRIC_RE = re.compile(r'mse:([\d.e-]+), mae:([\d.e-]+)')


def run(run_id, script, args, paper):
    run_dir = f'/content/runs/{run_id}'
    os.makedirs(run_dir, exist_ok=False)
    command = ['python', '-u', '/content/compat_run.py', script] + args
    start = time.time()
    with open(run_dir + '/train.log', 'w') as log:
        process = subprocess.Popen(command, cwd=run_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            log.write(line)
            if 'Epoch' in line or 'mse:' in line or 'Error' in line or 'Traceback' in line:
                print(line, end='')
        code = process.wait()
    elapsed = time.time() - start
    text = open(run_dir + '/train.log').read()
    epochs = [tuple(float(x) for x in m) for m in EPOCH_RE.findall(text)]
    found = METRIC_RE.findall(text)
    mse, mae = (float(found[-1][0]), float(found[-1][1])) if found else (None, None)
    summary = {
        'run_id': run_id, 'exit_code': code, 'elapsed_seconds': round(elapsed, 1),
        'command': command, 'mse': mse, 'mae': mae, 'paper_mse': paper[0], 'paper_mae': paper[1],
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'python': sys.version.split()[0], 'torch': torch.__version__,
        'numpy': numpy.__version__, 'pandas': pandas.__version__,
        'upstream_commit': UPSTREAM_COMMIT, 'epochs_logged': len(epochs),
    }
    if mse is not None:
        summary['mse_diff_pct'] = round(100 * (mse - paper[0]) / paper[0], 2)
        summary['mae_diff_pct'] = round(100 * (mae - paper[1]) / paper[1], 2)
    subprocess.run(f'pip freeze > {run_dir}/environment.txt', shell=True)
    json.dump(summary, open(run_dir + '/summary.json', 'w'), indent=2)
    print(json.dumps({k: summary.get(k) for k in ('exit_code', 'elapsed_seconds', 'mse', 'mae', 'mse_diff_pct', 'mae_diff_pct')}))
    if clearml_ok:
        from clearml import Task
        task = Task.init(project_name='D2 DLinear', task_name=run_id, task_type=Task.TaskTypes.training,
                         reuse_last_task_id=False, auto_connect_frameworks=False,
                         auto_connect_arg_parser=False, auto_resource_monitoring=False)
        task.connect_configuration({'command': ' '.join(command)}, name='command')
        logger = task.get_logger()
        for epoch, train, vali, test in epochs:
            logger.report_scalar('loss', 'train', train, iteration=int(epoch))
            logger.report_scalar('loss', 'vali', vali, iteration=int(epoch))
            logger.report_scalar('loss', 'test', test, iteration=int(epoch))
        if mse is not None:
            logger.report_single_value('mse', mse)
            logger.report_single_value('mae', mae)
        task.upload_artifact('summary', artifact_object=run_dir + '/summary.json')
        task.upload_artifact('train_log', artifact_object=run_dir + '/train.log')
        task.upload_artifact('environment', artifact_object=run_dir + '/environment.txt')
        task.flush(wait_for_uploads=True)
        print('clearml:', task.get_output_log_web_page())
        task.close()
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        subprocess.run(['mkdir', '-p', DRIVE_DIR])
        subprocess.run(['rsync', '-a', '--exclude', 'checkpoints', run_dir, DRIVE_DIR])
    return summary

### FEDformer, ETTh1, 96 → 96

Аргументы взяты из блока ETTh1 в `upstream/FEDformer/scripts/LongForecasting.sh`, кроме `--features M`. Сид авторский (2021, зашит в `run.py`), `num_workers` по умолчанию.

In [ ]:
fed_args = [
    '--is_training', '1', '--root_path', ROOT + '/data', '--data_path', 'ETTh1.csv',
    '--task_id', 'ETTh1', '--model', 'FEDformer', '--data', 'ETTh1', '--features', 'M',
    '--seq_len', '96', '--label_len', '48', '--pred_len', '96',
    '--e_layers', '2', '--d_layers', '1', '--factor', '3',
    '--enc_in', '7', '--dec_in', '7', '--c_out', '7',
    '--des', 'Exp', '--d_model', '512', '--itr', '1',
]
fed = run('fedformer-asis-etth1-96-96', ROOT + '/upstream/FEDformer/run.py', fed_args, (0.376, 0.419))

### DLinear, ETTh1, 336 → 96, в том же окружении

Тот же конфиг, что в пилоте на маке. Нужен, чтобы сравнить мак CPU + torch 1.9 и Colab T4 + свежий torch на одной модели.

In [ ]:
config = json.load(open(ROOT + '/configs/etth1-336-96.json'))
dl_args = []
for key, value in config.items():
    dl_args += ['--' + key, str(value)]
dl_args += ['--root_path', ROOT + '/data']
dl = run('dlinear-asis-etth1-336-96-colab', ROOT + '/upstream/run_longExp.py', dl_args, (0.375, 0.399))

## Сводка для pre-registration

In [ ]:
import numpy as np
for s in (fed, dl):
    print(s['run_id'])
    for key in ('gpu', 'python', 'torch', 'numpy', 'pandas', 'elapsed_seconds', 'epochs_logged', 'mse', 'mae', 'paper_mse', 'paper_mae', 'mse_diff_pct', 'mae_diff_pct'):
        print(' ', key, s.get(key))

pred_files = !find /content/runs/fedformer-asis-etth1-96-96/results -name pred.npy
fed_windows = np.load(pred_files[0], mmap_mode='r').shape[0]
print('fedformer: оценено тестовых окон', fed_windows, 'из 2785 (drop_last=True в коде FEDformer)')
print('dlinear на маке: mse 0.3841443955898285, mae 0.40471312403678894')